# 🟢 START TRAINING — Sesi 1 (Batas Waktu: 5 Jam)
> Notebook ini memulai training dari awal. Checkpoint otomatis disimpan sebelum waktu habis.
> Lanjutkan dengan notebook `2-balinese-whisper-resume.ipynb` di sesi berikutnya.

## Install Dependencies

In [ ]:
!pip install evaluate jiwer -q

## Get Dataset

In [ ]:
!git clone https://huggingface.co/datasets/Sparkplugx1904/Balinese-Common-Voice/ temp
!mv temp/* ./
!rm -rf temp

## Import Library

In [ ]:
from datasets import load_dataset, Audio, Features, Value, Dataset
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
import torch
import pandas as pd
import librosa

## Data Loading and Processing

In [ ]:
import pandas as pd
from datasets import Dataset, Value
from sklearn.model_selection import train_test_split

# 1. Load metadata
metadata_df = pd.read_csv("metadata.tsv", sep='\t')
metadata_df = metadata_df[["path", "balinese"]].rename(columns={"balinese": "sentence"})

if not metadata_df["path"].str.startswith("clips/").all():
    metadata_df["path"] = "clips/" + metadata_df["path"].astype(str)

metadata_df = metadata_df.dropna(subset=["path", "sentence"]).reset_index(drop=True)

# 2. Split
train_df, test_df = train_test_split(metadata_df, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Total data: {len(metadata_df)}")
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

# 3. Konversi ke HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset  = Dataset.from_pandas(test_df,  preserve_index=False)
train_dataset = train_dataset.cast_column("path", Value("string"))
test_dataset  = test_dataset.cast_column("path",  Value("string"))

print(train_dataset)

## Load Whisper Processor

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-medium")
tokenizer  = WhisperTokenizer.from_pretrained("openai/whisper-medium", language="indonesian", task="transcribe")
processor  = WhisperProcessor.from_pretrained("openai/whisper-medium", language="indonesian", task="transcribe")

def prepare_dataset(batch):
    batch["labels"] = processor.tokenizer(batch["sentence"], truncation=True).input_ids
    return batch

print("Melakukan tokenisasi dataset...")
train_dataset = train_dataset.map(prepare_dataset, remove_columns=["sentence"])
test_dataset  = test_dataset.map(prepare_dataset,  remove_columns=["sentence"])
print(f"Kolom: {train_dataset.column_names}")

## Load Pre-Trained Checkpoint

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-medium")
model.generation_config.forced_decoder_ids = None
model.generation_config.suppress_tokens    = []
print("Model loaded: openai/whisper-medium")

## Collator

In [ ]:
import librosa
import numpy as np
import torch

class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor
        self.sr = 16000
        self.pad = processor.tokenizer.pad_token_id

    def __call__(self, features):
        input_features = [{
            "input_features": self.processor.feature_extractor(
                librosa.load(feature["path"], sr=self.sr)[0],
                sampling_rate=self.sr
            ).input_features[0]
        } for feature in features]

        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        labels = [torch.tensor(feature["labels"]) for feature in features]
        labels_padded = torch.nn.utils.rnn.pad_sequence(
            labels, batch_first=True, padding_value=self.pad
        )
        batch["labels"] = labels_padded
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

## Metric (WER)

In [ ]:
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids  = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

print("Metric WER siap.")

## 🚀 Launch Training (Sesi 1 — Maks 5 Jam)

In [ ]:
import os
import gc
import time
import torch
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, TrainerCallback

torch.cuda.empty_cache()
gc.collect()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
model.config.use_cache = False

# ============================================================
# ⏱️ CALLBACK: Stop otomatis 15 menit sebelum batas 5 jam
#    agar ada cukup waktu menyimpan checkpoint dengan aman
# ============================================================
MAX_HOURS   = 5
SAFETY_MIN  = 15   # simpan 15 menit sebelum habis

class TimeLimitCallback(TrainerCallback):
    def __init__(self, max_hours, safety_minutes):
        self.deadline = time.time() + (max_hours * 3600) - (safety_minutes * 60)

    def on_step_end(self, args, state, control, **kwargs):
        # Bersihkan memori tiap 20 step
        if state.global_step % 20 == 0:
            torch.cuda.empty_cache()
            gc.collect()
        # Hentikan training jika mendekati batas waktu
        if time.time() >= self.deadline:
            print(f"\n⏰ Batas waktu {MAX_HOURS} jam hampir habis. Menghentikan training...")
            print(f"   Checkpoint terakhir akan disimpan di: ./whisper-balinese")
            control.should_training_stop = True
        return control

    def on_train_end(self, args, state, control, **kwargs):
        sisa = max(0, self.deadline - time.time())
        print(f"\n✅ Training dihentikan pada step {state.global_step}, epoch {state.epoch:.2f}")
        print(f"   Sisa waktu (buffer): {sisa/60:.1f} menit")
        print(f"   ➡️  Lanjutkan dengan notebook: 2-balinese-whisper-resume.ipynb")

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-balinese",

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,

    learning_rate=1e-5,
    warmup_steps=100,
    num_train_epochs=30,
    lr_scheduler_type="cosine",
    weight_decay=0.01,

    fp16=True,
    gradient_checkpointing=True,

    predict_with_generate=True,
    generation_max_length=225,
    eval_accumulation_steps=1,

    eval_strategy="steps",
    logging_steps=5,
    save_strategy="steps",
    save_steps=200,          # Simpan lebih sering agar checkpoint selalu tersedia
    eval_steps=200,
    report_to=["tensorboard"],

    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    save_total_limit=2,      # Simpan 2 checkpoint terakhir untuk keamanan resume
    remove_unused_columns=False,
    dataloader_num_workers=2,
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[TimeLimitCallback(MAX_HOURS, SAFETY_MIN)]
)

print("🟢 Memulai Training Sesi 1...")
print(f"⏱️  Akan berjalan maks {MAX_HOURS} jam (stop {SAFETY_MIN} menit sebelum batas)")
print(f"   Learning Rate : {training_args.learning_rate}")
print(f"   Total Epochs  : {training_args.num_train_epochs}")
print()

trainer.train()

## 💾 Simpan Checkpoint Sesi 1

In [ ]:
import os

# Simpan checkpoint sesi ini
trainer.save_model("./whisper-balinese")
processor.save_pretrained("./whisper-balinese")
tokenizer.save_pretrained("./whisper-balinese")

# Tampilkan info checkpoint yang tersimpan
checkpoints = sorted([d for d in os.listdir("./whisper-balinese") if d.startswith("checkpoint-")])
print("✅ Checkpoint tersimpan:")
for ck in checkpoints:
    print(f"   📁 ./whisper-balinese/{ck}")
print()
print("➡️  Upload folder './whisper-balinese' ke Kaggle Dataset / HuggingFace Hub")
print("   lalu gunakan di notebook: 2-balinese-whisper-resume.ipynb")